# Week 3 — Clean and normalize datasets

Cleans the raw financial, feature-matrix, and review datasets collected in Week 2.

**Input:** `data/raw/{play_store_reviews.json, app_store_reviews.json, cost_efficiency.csv, feature_matrix.csv}`

**Output:** `data/processed/{reviews_clean.csv, cost_efficiency_clean.csv, feature_matrix_clean.csv}`

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

## Reviews: merge, dedupe, normalize types

In [2]:
play = json.loads((RAW / "play_store_reviews.json").read_text(encoding="utf-8"))
app = json.loads((RAW / "app_store_reviews.json").read_text(encoding="utf-8"))
reviews_df = pd.concat([pd.DataFrame(play), pd.DataFrame(app)], ignore_index=True)

reviews_df["text"] = (
    reviews_df["text"].fillna("").astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
)
reviews_df["rating"] = pd.to_numeric(reviews_df["rating"], errors="coerce")
reviews_df["date"] = (
    pd.to_datetime(reviews_df["date"], errors="coerce", utc=True).dt.tz_localize(None)
)

before = len(reviews_df)
reviews_df = reviews_df[reviews_df["text"].str.len() > 0]
reviews_df = reviews_df.dropna(subset=["rating"])
reviews_df = reviews_df.drop_duplicates(subset=["entity", "platform", "text", "rating"])
reviews_df["rating"] = reviews_df["rating"].astype(int)

keep_cols = ["entity", "platform", "review_id", "rating", "text", "date", "app_version", "title"]
for c in keep_cols:
    if c not in reviews_df.columns:
        reviews_df[c] = np.nan
reviews_df = reviews_df[keep_cols].sort_values(["entity", "platform", "date"]).reset_index(drop=True)

reviews_df.to_csv(PROCESSED / "reviews_clean.csv", index=False)
print(f"reviews: {before} raw -> {len(reviews_df)} clean rows -> data/processed/reviews_clean.csv")
reviews_df.head()

reviews: 1400 raw -> 1302 clean rows -> data/processed/reviews_clean.csv


,entity,platform,review_id,rating,text,date,app_version,title
0,hdfc_bank,app_store,14285276560,5,Great,NaT,NaN,Rating
1,hdfc_bank,app_store,14285125471,5,It’s better app,NaT,NaN,Good one
2,hdfc_bank,app_store,14284893772,1,Very poor sarvice i think i currant account i ...,NaT,NaN,Sarvice
3,hdfc_bank,app_store,14284815366,5,Easy handling,NaT,NaN,Good App
4,hdfc_bank,app_store,14284475135,1,new app very complicated,NaT,NaN,aa


## Cost efficiency: numeric coercion, whitespace/case normalization

Note: the raw CSV had two rows (`Jupiter Money Fintech (NBFC arm, standalone)`) with an
unescaped comma inside the `entity` field, which broke CSV parsing until quoted at the
source (`data/raw/cost_efficiency.csv`).

In [3]:
cost_df = pd.read_csv(RAW / "cost_efficiency.csv")
cost_df["entity"] = cost_df["entity"].str.strip()
cost_df["metric"] = cost_df["metric"].str.strip()
cost_df["value"] = pd.to_numeric(cost_df["value"], errors="coerce")
cost_df["confidence"] = cost_df["confidence"].str.strip().str.lower()
cost_df = cost_df.drop_duplicates()

cost_df.to_csv(PROCESSED / "cost_efficiency_clean.csv", index=False)
print(f"cost_efficiency: {len(cost_df)} rows -> data/processed/cost_efficiency_clean.csv")
cost_df.head()

cost_efficiency: 14 rows -> data/processed/cost_efficiency_clean.csv


,entity,metric,value,unit,fiscal_year,source,confidence
0,HDFC Bank,Cost-to-Income Ratio,40.20,%,FY2023-24,HDFC Bank Q4FY24 investor presentation,official
1,HDFC Bank,Operating (non-interest) expenses,63386.00,INR crore,FY2023-24,Equitymaster FY24 annual report analysis,official
2,HDFC Bank,Total operating income (interest+other),407995.00,INR crore,FY2023-24,Equitymaster FY24 annual report analysis,official
3,HDFC Bank,Net profit,64062.00,INR crore,FY2023-24,Equitymaster FY24 annual report analysis,official
4,Jupiter (Amica Financial Technologies),Revenue from operations (parent),35.85,INR crore,FY2023-24,Entrackr Nov 2024,official (roc filing via press)


## Feature matrix: whitespace/case normalization

In [4]:
feature_df = pd.read_csv(RAW / "feature_matrix.csv")
feature_df["feature"] = feature_df["feature"].str.strip()
for col in ["jupiter", "hdfc_bank"]:
    feature_df[col] = feature_df[col].str.strip().str.title()

feature_df.to_csv(PROCESSED / "feature_matrix_clean.csv", index=False)
print(f"feature_matrix: {len(feature_df)} rows -> data/processed/feature_matrix_clean.csv")
feature_df

feature_matrix: 10 rows -> data/processed/feature_matrix_clean.csv


,feature,jupiter,hdfc_bank,notes_source
0,Savings Account,Yes,Yes,Jupiter: digital-only account issued via Feder...
1,Fixed Deposit (FD),Yes,Yes,"Jupiter: FDs issued by partner Federal Bank, b..."
2,Debit Card,Yes,Yes,Jupiter: Federal Bank debit card with cashback...
3,Credit Card,Partial,Yes,Jupiter: only a co-branded RuPay Credit Card (...
4,UPI,Yes,Yes,"Jupiter: UPI + RuPay Credit Card UPI linking, ..."
5,Personal Loans/Lending,Partial,Yes,Jupiter: 0% 'Mini Loans' up to INR 50k for sal...
6,Investments (MF/Stocks),Partial,Yes,Jupiter: Mutual Funds and Digital Gold (via MM...
7,Insurance,Partial,Partial,Jupiter: health insurance launched since 2025 ...
8,Forex/International Transactions,Partial,Yes,Jupiter: 3.5% flat forex fee on international ...
9,Rewards/Cashback,Yes,Yes,"Jupiter: 1% reward on debit/UPI spends, up to ..."
